# Goal Post Shifting Score (GPSS) Pipeline

## Technical Spec & Executable Notebook

**Purpose:** Detect pre-earnings-miss linguistic drift in SaaS company earnings calls using an LLM-as-Judge approach.

**Hypothesis (H*):** Firms that experience a negative earnings outcome in quarter *t* exhibit higher Goal Post Shifting Scores (GPSS) in their earnings call transcripts from quarter *t−1* or *t−2*, relative to firm-quarters not followed by a negative outcome.

**Pipeline Stages:**
1. Configuration & universe definition
2. Data ingestion (transcripts, earnings, prices) via Alpha Vantage API
3. Outcome labeling (MISS / NO_MISS)
4. GPSS scoring via structured LLM rubric
5. Temporal alignment (score at *t* → outcome at *t+1*)
6. Statistical testing & visualization

**Data Source:** Alpha Vantage API (single provider for all three data streams)

---

## 0. Environment Setup

Run once to install all required dependencies.

In [ ]:
!pip install requests pandas numpy scipy statsmodels matplotlib seaborn tqdm anthropic -q

## 1. Configuration

All configurable parameters live in this single cell. Edit API keys and universe here.

In [ ]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
from tqdm.notebook import tqdm

# ── API Keys ──────────────────────────────────────────────────────────────────
ALPHA_VANTAGE_API_KEY = os.environ.get("ALPHA_VANTAGE_API_KEY", "YOUR_KEY_HERE")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE")

# ── SaaS Universe ─────────────────────────────────────────────────────────────
# Subscription-heavy software firms with 5+ years of public earnings calls.
# Chosen for: regular guidance issuance, analyst coverage, flexible KPI culture.
UNIVERSE = [
    "CRM",   # Salesforce
    "NOW",   # ServiceNow
    "WDAY",  # Workday
    "ZS",    # Zscaler
    "DDOG",  # Datadog
    "NET",   # Cloudflare
    "SNOW",  # Snowflake
    "MDB",   # MongoDB
    "PANW",  # Palo Alto Networks
    "CRWD",  # CrowdStrike
    "ZM",    # Zoom
    "OKTA",  # Okta
]

# ── Time Range ────────────────────────────────────────────────────────────────
# Generate quarter labels: 2020Q1 through 2025Q3
START_YEAR = 2020
END_YEAR = 2025
END_QUARTER = 3  # inclusive

QUARTERS = []
for y in range(START_YEAR, END_YEAR + 1):
    max_q = END_QUARTER if y == END_YEAR else 4
    for q in range(1, max_q + 1):
        QUARTERS.append(f"{y}Q{q}")

print(f"Universe: {len(UNIVERSE)} tickers")
print(f"Quarters: {len(QUARTERS)} ({QUARTERS[0]} → {QUARTERS[-1]})")
print(f"Max observations: {len(UNIVERSE) * len(QUARTERS)}")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
for d in [RAW_DIR / "transcripts", RAW_DIR / "earnings", RAW_DIR / "prices", PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Rate Limiting ─────────────────────────────────────────────────────────────
# Free tier: 25 requests/day, ~5/min. Paid tier: much higher.
# Set to 12 seconds between calls for free tier safety. Reduce if on paid plan.
API_DELAY_SECONDS = 12

# ── LLM Config ────────────────────────────────────────────────────────────────
LLM_MODEL = "claude-sonnet-4-20250514"  # balance of cost and quality
LLM_TEMPERATURE = 0.0  # deterministic scoring

## 2. Alpha Vantage API Helper

Thin wrapper with caching and rate limiting. Every API response is saved to disk so re-runs don't burn API calls.

In [ ]:
AV_BASE = "https://www.alphavantage.co/query"

def av_request(params: dict, cache_path: Path) -> dict:
    """
    Make an Alpha Vantage API request with file-based caching.
    Returns parsed JSON. Skips API call if cache file exists.
    """
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    params["apikey"] = ALPHA_VANTAGE_API_KEY
    resp = requests.get(AV_BASE, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    # Check for rate limit / error messages
    if "Information" in data and "rate limit" in data["Information"].lower():
        print(f"  ⚠️  Rate limited. Waiting 60s...")
        time.sleep(60)
        return av_request(params, cache_path)  # retry once

    if "Error Message" in data:
        print(f"  ❌ API error: {data['Error Message']}")
        return data

    # Cache successful response
    with open(cache_path, "w") as f:
        json.dump(data, f)

    time.sleep(API_DELAY_SECONDS)
    return data

print("✅ API helper ready")

## 3. Ingest Earnings Data

Pulls quarterly EPS actuals, estimates, and surprise metrics for each ticker.  
**Cost:** 1 API call per ticker (returns full history).

In [ ]:
def fetch_earnings(symbol: str) -> pd.DataFrame:
    """Fetch quarterly earnings (EPS actual vs estimate) for a symbol."""
    cache = RAW_DIR / "earnings" / f"{symbol}.json"
    data = av_request({"function": "EARNINGS", "symbol": symbol}, cache)

    quarterly = data.get("quarterlyEarnings", [])
    if not quarterly:
        print(f"  ⚠️  No earnings data for {symbol}")
        return pd.DataFrame()

    df = pd.DataFrame(quarterly)
    df["symbol"] = symbol

    # Type conversions
    for col in ["reportedEPS", "estimatedEPS", "surprise", "surprisePercentage"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["reportedDate"] = pd.to_datetime(df["reportedDate"], errors="coerce")
    df["fiscalDateEnding"] = pd.to_datetime(df["fiscalDateEnding"], errors="coerce")

    return df

# ── Run ingestion ─────────────────────────────────────────────────────────────
earnings_frames = []
for sym in tqdm(UNIVERSE, desc="Fetching earnings"):
    df = fetch_earnings(sym)
    if not df.empty:
        earnings_frames.append(df)

earnings_df = pd.concat(earnings_frames, ignore_index=True)
earnings_df.to_parquet(PROCESSED_DIR / "earnings.parquet", index=False)
print(f"\n✅ Earnings: {len(earnings_df)} quarterly observations across {earnings_df['symbol'].nunique()} tickers")
earnings_df.head()

## 4. Ingest Earnings Call Transcripts

Pulls full transcript text for each ticker × quarter.  
**Cost:** 1 API call per ticker-quarter. This is the most expensive step.  

**Rate limit strategy:** The loop caches every successful response. If you hit the daily limit, just re-run the cell the next day — it skips already-cached transcripts.

> ⚡ **Tip:** On the free tier (25/day), set `MAX_TRANSCRIPTS_PER_RUN` to control burn rate.

In [ ]:
MAX_TRANSCRIPTS_PER_RUN = 25  # set to None for unlimited (paid tier)

def fetch_transcript(symbol: str, quarter: str) -> dict | None:
    """Fetch earnings call transcript. Returns raw JSON or None."""
    cache = RAW_DIR / "transcripts" / f"{symbol}_{quarter}.json"
    data = av_request(
        {"function": "EARNINGS_CALL_TRANSCRIPT", "symbol": symbol, "quarter": quarter},
        cache
    )
    transcript = data.get("transcript", [])
    if not transcript:
        return None
    return data

def parse_transcript(data: dict) -> dict:
    """
    Parse raw transcript JSON into structured components:
    - full_text: concatenated transcript
    - prepared_remarks: speaker turns before first analyst
    - qa_section: speaker turns from first analyst onward
    - speakers: list of {speaker, title, content} dicts
    """
    speakers = data.get("transcript", [])
    if isinstance(speakers, str):
        # Sometimes returned as string; try parse
        try:
            speakers = json.loads(speakers)
        except json.JSONDecodeError:
            return {"full_text": speakers, "prepared_remarks": speakers,
                    "qa_section": "", "speakers": []}

    full_text_parts = []
    prepared_parts = []
    qa_parts = []
    in_qa = False

    for turn in speakers:
        title = turn.get("title", "").lower()
        content = turn.get("content", "")
        speaker = turn.get("speaker", "")
        label = f"{speaker} ({turn.get('title', '')}):"

        full_text_parts.append(f"{label}\n{content}")

        # Heuristic: first "Analyst" title marks start of Q&A
        if not in_qa and "analyst" in title:
            in_qa = True

        if in_qa:
            qa_parts.append(f"{label}\n{content}")
        else:
            prepared_parts.append(f"{label}\n{content}")

    return {
        "full_text": "\n\n".join(full_text_parts),
        "prepared_remarks": "\n\n".join(prepared_parts),
        "qa_section": "\n\n".join(qa_parts),
        "speakers": speakers,
    }

# ── Run ingestion ─────────────────────────────────────────────────────────────
transcript_records = []
api_calls_made = 0

for sym in UNIVERSE:
    for qtr in QUARTERS:
        # Check if already cached
        cache = RAW_DIR / "transcripts" / f"{sym}_{qtr}.json"
        is_cached = cache.exists()

        if not is_cached and MAX_TRANSCRIPTS_PER_RUN and api_calls_made >= MAX_TRANSCRIPTS_PER_RUN:
            continue  # skip to avoid burning quota

        data = fetch_transcript(sym, qtr)
        if not is_cached and data is not None:
            api_calls_made += 1

        if data is None:
            continue

        parsed = parse_transcript(data)
        transcript_records.append({
            "symbol": sym,
            "quarter": qtr,
            "full_text": parsed["full_text"],
            "prepared_remarks": parsed["prepared_remarks"],
            "qa_section": parsed["qa_section"],
            "n_speakers": len(parsed["speakers"]),
            "char_count": len(parsed["full_text"]),
        })

transcripts_df = pd.DataFrame(transcript_records)
if not transcripts_df.empty:
    transcripts_df.to_parquet(PROCESSED_DIR / "transcripts.parquet", index=False)

print(f"\n✅ Transcripts: {len(transcripts_df)} collected")
print(f"   API calls this run: {api_calls_made}")
print(f"   Coverage: {transcripts_df['symbol'].nunique() if not transcripts_df.empty else 0} tickers")
if not transcripts_df.empty:
    print(f"   Avg length: {transcripts_df['char_count'].mean():,.0f} chars")
    print(transcripts_df.groupby("symbol").size().to_string())

## 5. Ingest Daily Stock Prices

Pulls adjusted daily OHLCV for each ticker + SPY (S&P 500 benchmark).  
**Cost:** 1 API call per ticker (returns full history).

In [ ]:
def fetch_daily_prices(symbol: str) -> pd.DataFrame:
    """Fetch daily adjusted close prices for a symbol."""
    cache = RAW_DIR / "prices" / f"{symbol}.json"
    data = av_request(
        {"function": "TIME_SERIES_DAILY_ADJUSTED", "symbol": symbol, "outputsize": "full"},
        cache
    )

    ts_key = "Time Series (Daily)"
    if ts_key not in data:
        print(f"  ⚠️  No price data for {symbol}")
        return pd.DataFrame()

    records = []
    for date_str, ohlcv in data[ts_key].items():
        records.append({
            "date": pd.to_datetime(date_str),
            "symbol": symbol,
            "open": float(ohlcv["1. open"]),
            "high": float(ohlcv["2. high"]),
            "low": float(ohlcv["3. low"]),
            "close": float(ohlcv["4. close"]),
            "adjusted_close": float(ohlcv["5. adjusted close"]),
            "volume": int(ohlcv["6. volume"]),
        })

    df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
    return df

# ── Fetch all tickers + SPY benchmark ─────────────────────────────────────────
price_tickers = UNIVERSE + ["SPY"]
price_frames = []

for sym in tqdm(price_tickers, desc="Fetching prices"):
    df = fetch_daily_prices(sym)
    if not df.empty:
        price_frames.append(df)

prices_df = pd.concat(price_frames, ignore_index=True)
prices_df.to_parquet(PROCESSED_DIR / "prices.parquet", index=False)
print(f"\n✅ Prices: {len(prices_df)} daily bars across {prices_df['symbol'].nunique()} tickers")
print(f"   Date range: {prices_df['date'].min().date()} → {prices_df['date'].max().date()}")

## 6. Outcome Labeling

For each firm-quarter, compute a binary **MISS** / **NO_MISS** label using two criteria:

| Criterion | Source | Threshold |
|-----------|--------|-----------|
| EPS surprise | `EARNINGS` endpoint | `surprise < 0` |
| Abnormal return | Daily prices − SPY | `CAR[0, +3] ≤ −5%` |

**MISS** = either criterion is true.  
**NO_MISS** = EPS beat/met AND abnormal return > −5%.

In [ ]:
def compute_abnormal_return(symbol: str, report_date: pd.Timestamp,
                             prices: pd.DataFrame, window: int = 3) -> float | None:
    """
    Compute cumulative abnormal return (CAR) over [0, +window] trading days
    around the report_date. Abnormal = firm return − SPY return.
    """
    firm_prices = prices[prices["symbol"] == symbol].set_index("date")["adjusted_close"].sort_index()
    spy_prices = prices[prices["symbol"] == "SPY"].set_index("date")["adjusted_close"].sort_index()

    # Find the trading day on or after report_date (earnings may report after close)
    valid_dates = firm_prices.index[firm_prices.index >= report_date]
    if len(valid_dates) < window + 1:
        return None

    t0 = valid_dates[0]
    t_end = valid_dates[window]

    # Need the day before t0 for return calc
    prior_dates = firm_prices.index[firm_prices.index < t0]
    if len(prior_dates) == 0:
        return None
    t_minus1 = prior_dates[-1]

    try:
        firm_ret = (firm_prices[t_end] / firm_prices[t_minus1]) - 1
        spy_ret = (spy_prices[t_end] / spy_prices[t_minus1]) - 1
        return firm_ret - spy_ret
    except (KeyError, ZeroDivisionError):
        return None

# ── Build labels ──────────────────────────────────────────────────────────────
prices = pd.read_parquet(PROCESSED_DIR / "prices.parquet")
earnings = pd.read_parquet(PROCESSED_DIR / "earnings.parquet")

label_records = []
for _, row in tqdm(earnings.iterrows(), total=len(earnings), desc="Labeling outcomes"):
    if pd.isna(row["reportedDate"]):
        continue

    car = compute_abnormal_return(row["symbol"], row["reportedDate"], prices)
    eps_miss = (row["surprise"] < 0) if pd.notna(row["surprise"]) else None
    car_miss = (car <= -0.05) if car is not None else None

    # MISS if either criterion is true
    if eps_miss is None and car_miss is None:
        outcome = None
    else:
        outcome = bool((eps_miss is True) or (car_miss is True))

    label_records.append({
        "symbol": row["symbol"],
        "fiscalDateEnding": row["fiscalDateEnding"],
        "reportedDate": row["reportedDate"],
        "reportedEPS": row["reportedEPS"],
        "estimatedEPS": row["estimatedEPS"],
        "surprise": row["surprise"],
        "surprisePercentage": row["surprisePercentage"],
        "car_0_3": car,
        "eps_miss": eps_miss,
        "car_miss": car_miss,
        "outcome_miss": outcome,
    })

labels_df = pd.DataFrame(label_records).dropna(subset=["outcome_miss"])
labels_df.to_parquet(PROCESSED_DIR / "labels.parquet", index=False)

n_miss = labels_df["outcome_miss"].sum()
n_total = len(labels_df)
print(f"\n✅ Labels: {n_total} firm-quarters")
print(f"   MISS: {n_miss} ({100*n_miss/n_total:.1f}%)")
print(f"   NO_MISS: {n_total - n_miss} ({100*(n_total-n_miss)/n_total:.1f}%)")

## 7. GPSS Scoring via LLM

This is the core innovation. Each transcript is evaluated by the LLM on 6 dimensions:

| Dim | Name | What it Detects |
|-----|------|----------------|
| D1 | Metric Substitution | Hard KPIs → soft/adjusted metrics |
| D2 | Timeline Deferral | Near-term → "longer-term" framing |
| D3 | Guidance Vagueness | Precise ranges → qualitative language |
| D4 | Hedging Escalation | Increased conditional/uncertain language |
| D5 | External Attribution | Blame shifted to macro/FX/supply chain |
| D6 | Q&A Deflection | Analyst questions redirected or non-answered |

Each dimension is scored 1–5. Composite GPSS = mean(D1–D6).

> ⚡ **Cost estimate:** ~300 transcripts × ~15k tokens each × Claude Sonnet ≈ $30–60 in API fees.

In [ ]:
GPSS_SYSTEM_PROMPT = """
You are an expert financial analyst evaluating an earnings call transcript for signs of
"goal post shifting" — subtle changes in how management frames performance expectations
that may precede disappointing results.

Score each dimension on a 1–5 integer scale:
  1 = No evidence of shifting. Communication is direct, specific, and consistent.
  2 = Minimal. Mostly direct with occasional soft language.
  3 = Moderate. Noticeable hedging or reframing on some topics.
  4 = Significant. Multiple clear instances of shifting behavior.
  5 = Pervasive. Management systematically reframes, deflects, or obfuscates.

DIMENSIONS:

D1 (Metric Substitution): Does management de-emphasize standard financial KPIs
   (revenue, EPS, operating margin) in favor of adjusted, non-standard, or proxy
   metrics (ARR, adjusted EBITDA, "Rule of 40", engagement metrics)?
   - Score 1: Leads with GAAP revenue and EPS, discusses them thoroughly.
   - Score 5: Barely mentions revenue/EPS; pivots to non-standard metrics.

D2 (Timeline Deferral): Does management shift performance targets from near-term
   ("this quarter," "this year") to longer-term horizons ("over time," "multi-year")?
   - Score 1: Clear near-term commitments with specific timelines.
   - Score 5: All targets framed as long-term; near-term specifics avoided.

D3 (Guidance Vagueness): Has forward-looking guidance become less precise?
   Compare to what an analyst would expect (specific ranges vs. qualitative).
   - Score 1: Tight numeric ranges provided for key metrics.
   - Score 5: No numeric guidance; only directional or qualitative commentary.

D4 (Hedging Escalation): Is there elevated use of hedging and uncertainty language?
   ("approximately," "subject to," "depending on," "we expect," "roughly")
   - Score 1: Confident, definitive statements.
   - Score 5: Nearly every forward statement heavily hedged.

D5 (External Attribution): Does management attribute current or expected performance
   to external/uncontrollable factors (macro, FX, supply chain, regulatory timing)?
   - Score 1: Takes ownership of results; external factors mentioned in passing.
   - Score 5: External factors dominate the narrative; little ownership of results.

D6 (Q&A Deflection): In the Q&A section, does management redirect analyst questions
   to unrelated topics, give non-specific responses, or pivot to long-term strategy?
   - Score 1: Direct, specific answers that address the analyst's question.
   - Score 5: Consistently redirects, gives vague answers, or filibusters.

RESPONSE FORMAT — return ONLY valid JSON, no markdown, no preamble:
{
  "D1": <int 1-5>,
  "D2": <int 1-5>,
  "D3": <int 1-5>,
  "D4": <int 1-5>,
  "D5": <int 1-5>,
  "D6": <int 1-5>,
  "rationale": {
    "D1": "<1-2 sentence justification>",
    "D2": "<1-2 sentence justification>",
    "D3": "<1-2 sentence justification>",
    "D4": "<1-2 sentence justification>",
    "D5": "<1-2 sentence justification>",
    "D6": "<1-2 sentence justification>"
  }
}
""".strip()

print("✅ GPSS prompt defined")
print(f"   System prompt: {len(GPSS_SYSTEM_PROMPT)} chars")

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

GPSS_CACHE_DIR = DATA_DIR / "gpss_scores"
GPSS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def score_transcript(symbol: str, quarter: str, transcript_text: str) -> dict | None:
    """
    Score a single transcript using the GPSS rubric via Claude.
    Returns dict with D1-D6 scores and rationale, or None on failure.
    """
    cache_path = GPSS_CACHE_DIR / f"{symbol}_{quarter}.json"
    if cache_path.exists():
        with open(cache_path) as f:
            return json.load(f)

    # Truncate very long transcripts to ~80k chars (~20k tokens) to stay within limits
    max_chars = 80_000
    if len(transcript_text) > max_chars:
        # Keep first 60% (prepared remarks) + last 40% (Q&A end)
        split = int(max_chars * 0.6)
        transcript_text = (
            transcript_text[:split]
            + "\n\n[... transcript truncated for length ...]\n\n"
            + transcript_text[-(max_chars - split):]
        )

    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=1024,
            temperature=LLM_TEMPERATURE,
            system=GPSS_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": transcript_text}],
        )
        raw = response.content[0].text.strip()

        # Parse JSON (handle possible markdown fencing)
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]

        scores = json.loads(raw)

        # Validate
        for dim in ["D1", "D2", "D3", "D4", "D5", "D6"]:
            assert dim in scores, f"Missing {dim}"
            assert isinstance(scores[dim], int) and 1 <= scores[dim] <= 5, f"{dim} out of range"

        scores["symbol"] = symbol
        scores["quarter"] = quarter
        scores["gpss_composite"] = np.mean([scores[d] for d in ["D1","D2","D3","D4","D5","D6"]])

        # Cache
        with open(cache_path, "w") as f:
            json.dump(scores, f, indent=2)

        return scores

    except Exception as e:
        print(f"  ❌ Error scoring {symbol} {quarter}: {e}")
        return None

print("✅ Scoring function ready")

In [ ]:
# ── Score all transcripts ─────────────────────────────────────────────────────
transcripts = pd.read_parquet(PROCESSED_DIR / "transcripts.parquet")

score_records = []
for _, row in tqdm(transcripts.iterrows(), total=len(transcripts), desc="Scoring transcripts"):
    result = score_transcript(row["symbol"], row["quarter"], row["full_text"])
    if result:
        score_records.append(result)

scores_df = pd.DataFrame(score_records)
if not scores_df.empty:
    scores_df.to_parquet(PROCESSED_DIR / "gpss_scores.parquet", index=False)

print(f"\n✅ Scored {len(scores_df)} transcripts")
if not scores_df.empty:
    print(f"   Mean GPSS: {scores_df['gpss_composite'].mean():.2f}")
    print(f"   Std GPSS:  {scores_df['gpss_composite'].std():.2f}")
    print("\n   Dimension means:")
    for d in ["D1","D2","D3","D4","D5","D6"]:
        print(f"     {d}: {scores_df[d].mean():.2f}")

## 8. Temporal Alignment & Merge

The critical step: align **GPSS from quarter *t*** with **outcome from quarter *t+1***.

This tests the **leading-indicator** property — does language *before* the miss predict the miss?

In [ ]:
def quarter_to_sortable(q: str) -> int:
    """Convert '2023Q2' to sortable int 20232."""
    year, qnum = q.split("Q")
    return int(year) * 10 + int(qnum)

def next_quarter(q: str) -> str:
    """Return the next calendar quarter label."""
    year, qnum = int(q[:4]), int(q[-1])
    if qnum == 4:
        return f"{year+1}Q1"
    return f"{year}Q{qnum+1}"

# Load data
scores = pd.read_parquet(PROCESSED_DIR / "gpss_scores.parquet")
labels = pd.read_parquet(PROCESSED_DIR / "labels.parquet")

# Map labels to fiscal quarters (approximate: fiscal date ending → calendar quarter)
labels["cal_quarter"] = labels["fiscalDateEnding"].apply(
    lambda d: f"{d.year}Q{(d.month - 1) // 3 + 1}" if pd.notna(d) else None
)

# For each GPSS score at quarter t, look up outcome at t+1
scores["next_quarter"] = scores["quarter"].apply(next_quarter)

# Build outcome lookup: (symbol, cal_quarter) → outcome_miss
outcome_lookup = labels.set_index(["symbol", "cal_quarter"])["outcome_miss"].to_dict()

scores["outcome_t1"] = scores.apply(
    lambda r: outcome_lookup.get((r["symbol"], r["next_quarter"])), axis=1
)

# Z-score GPSS within each firm (controls for persistent communication style)
scores["gpss_z"] = scores.groupby("symbol")["gpss_composite"].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0
)

# Filter to observations where we have both score and next-quarter outcome
analysis_df = scores.dropna(subset=["outcome_t1"]).copy()
analysis_df["outcome_t1"] = analysis_df["outcome_t1"].astype(bool)
analysis_df.to_parquet(PROCESSED_DIR / "analysis_dataset.parquet", index=False)

print(f"✅ Analysis dataset: {len(analysis_df)} observations")
print(f"   Pre-MISS: {analysis_df['outcome_t1'].sum()}")
print(f"   Pre-NO_MISS: {(~analysis_df['outcome_t1']).sum()}")

## 9. Statistical Testing

Three tests of the null hypothesis H₀*:

1. **Welch's t-test** — are mean GPSS scores different between pre-miss and non-miss groups?
2. **Mann-Whitney U** — nonparametric rank test (no normality assumption)
3. **Logistic regression** — does GPSS predict MISS after controlling for past return?

In [ ]:
from scipy import stats
import statsmodels.api as sm

analysis = pd.read_parquet(PROCESSED_DIR / "analysis_dataset.parquet")

pre_miss = analysis[analysis["outcome_t1"] == True]["gpss_z"]
pre_nomiss = analysis[analysis["outcome_t1"] == False]["gpss_z"]

print("=" * 60)
print("HYPOTHESIS TEST RESULTS")
print("=" * 60)
print(f"\nSample sizes: pre-MISS = {len(pre_miss)}, pre-NO_MISS = {len(pre_nomiss)}")
print(f"Mean GPSS_z  : pre-MISS = {pre_miss.mean():.3f}, pre-NO_MISS = {pre_nomiss.mean():.3f}")
print(f"Median GPSS_z: pre-MISS = {pre_miss.median():.3f}, pre-NO_MISS = {pre_nomiss.median():.3f}")

# ── Test 1: Welch's t-test ────────────────────────────────────────────────────
t_stat, t_pval = stats.ttest_ind(pre_miss, pre_nomiss, equal_var=False)
print(f"\n1. Welch's t-test:")
print(f"   t-statistic = {t_stat:.3f}")
print(f"   p-value     = {t_pval:.4f}")
print(f"   {'✅ REJECT H₀ (p < 0.05)' if t_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# ── Test 2: Mann-Whitney U ────────────────────────────────────────────────────
u_stat, u_pval = stats.mannwhitneyu(pre_miss, pre_nomiss, alternative="greater")
print(f"\n2. Mann-Whitney U (one-sided: pre-miss > pre-nomiss):")
print(f"   U-statistic = {u_stat:.1f}")
print(f"   p-value     = {u_pval:.4f}")
print(f"   {'✅ REJECT H₀ (p < 0.05)' if u_pval < 0.05 else '❌ FAIL TO REJECT H₀'}")

# ── Test 3: Logistic Regression ───────────────────────────────────────────────
print(f"\n3. Logistic Regression: P(MISS_t+1) ~ GPSS_z(t)")
try:
    y = analysis["outcome_t1"].astype(int)
    X = sm.add_constant(analysis[["gpss_z"]])
    logit_model = sm.Logit(y, X).fit(disp=0)
    print(logit_model.summary2().tables[1].to_string())
    print(f"\n   Pseudo R²: {logit_model.prsquared:.4f}")
    coef = logit_model.params["gpss_z"]
    pval = logit_model.pvalues["gpss_z"]
    print(f"   GPSS_z coefficient: {coef:.3f} (p = {pval:.4f})")
    print(f"   {'✅ REJECT H₀' if pval < 0.05 else '❌ FAIL TO REJECT H₀'}")
except Exception as e:
    print(f"   ⚠️ Regression failed (likely insufficient data): {e}")

# ── Effect size (Cohen's d) ───────────────────────────────────────────────────
pooled_std = np.sqrt((pre_miss.std()**2 + pre_nomiss.std()**2) / 2)
cohens_d = (pre_miss.mean() - pre_nomiss.mean()) / pooled_std if pooled_std > 0 else 0
print(f"\nEffect size (Cohen's d): {cohens_d:.3f}")
print(f"   {'Large' if abs(cohens_d) > 0.8 else 'Medium' if abs(cohens_d) > 0.5 else 'Small' if abs(cohens_d) > 0.2 else 'Negligible'}")

## 10. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

analysis = pd.read_parquet(PROCESSED_DIR / "analysis_dataset.parquet")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Goal Post Shifting Score (GPSS) — Pre-Miss vs. Non-Miss", fontsize=14, fontweight="bold")

# ── Plot 1: GPSS_z distributions ──────────────────────────────────────────────
ax = axes[0, 0]
for label, color in [(True, "#e53e3e"), (False, "#3182ce")]:
    subset = analysis[analysis["outcome_t1"] == label]["gpss_z"]
    ax.hist(subset, bins=15, alpha=0.5, color=color,
            label=f"Pre-{'MISS' if label else 'NO_MISS'} (n={len(subset)})")
ax.set_xlabel("GPSS (z-scored within firm)")
ax.set_ylabel("Count")
ax.set_title("Distribution of GPSS_z by Next-Quarter Outcome")
ax.legend()

# ── Plot 2: Box plot by outcome ───────────────────────────────────────────────
ax = axes[0, 1]
analysis["outcome_label"] = analysis["outcome_t1"].map({True: "Pre-MISS", False: "Pre-NO_MISS"})
sns.boxplot(data=analysis, x="outcome_label", y="gpss_composite", ax=ax,
            palette={"Pre-MISS": "#e53e3e", "Pre-NO_MISS": "#3182ce"})
ax.set_title("Raw GPSS Composite by Next-Quarter Outcome")
ax.set_xlabel("")
ax.set_ylabel("GPSS Composite (1–5)")

# ── Plot 3: Subdimension comparison ───────────────────────────────────────────
ax = axes[1, 0]
dims = ["D1", "D2", "D3", "D4", "D5", "D6"]
dim_labels = ["Metric\nSubst.", "Timeline\nDeferral", "Guidance\nVagueness",
              "Hedging\nEscalation", "External\nAttribution", "Q&A\nDeflection"]
miss_means = [analysis[analysis["outcome_t1"]==True][d].mean() for d in dims]
nomiss_means = [analysis[analysis["outcome_t1"]==False][d].mean() for d in dims]
x = np.arange(len(dims))
w = 0.35
ax.bar(x - w/2, miss_means, w, label="Pre-MISS", color="#e53e3e", alpha=0.8)
ax.bar(x + w/2, nomiss_means, w, label="Pre-NO_MISS", color="#3182ce", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(dim_labels, fontsize=8)
ax.set_ylabel("Mean Score (1–5)")
ax.set_title("Subdimension Scores: Pre-MISS vs. Pre-NO_MISS")
ax.legend()
ax.set_ylim(0, 5)

# ── Plot 4: GPSS over time by firm ────────────────────────────────────────────
ax = axes[1, 1]
analysis["q_sort"] = analysis["quarter"].apply(quarter_to_sortable)
for sym in analysis["symbol"].unique()[:6]:  # top 6 for readability
    firm = analysis[analysis["symbol"] == sym].sort_values("q_sort")
    ax.plot(firm["quarter"], firm["gpss_composite"], marker="o", markersize=3, label=sym, alpha=0.7)
ax.set_title("GPSS Composite Over Time (sample firms)")
ax.set_xlabel("Quarter")
ax.set_ylabel("GPSS Composite")
ax.legend(fontsize=7, ncol=2)
ax.tick_params(axis="x", rotation=45, labelsize=7)

plt.tight_layout()
plt.savefig(PROCESSED_DIR / "gpss_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n✅ Plots saved to data/processed/gpss_results.png")

## 11. Validation Checks

Three robustness tests to confirm the signal isn't spurious.

In [ ]:
print("=" * 60)
print("VALIDATION TESTS")
print("=" * 60)

analysis = pd.read_parquet(PROCESSED_DIR / "analysis_dataset.parquet")

# ── Validation 1: Placebo Test (permutation) ─────────────────────────────────
print("\n1. PLACEBO TEST (1000 random permutations)")
observed_diff = (
    analysis[analysis["outcome_t1"]==True]["gpss_z"].mean()
    - analysis[analysis["outcome_t1"]==False]["gpss_z"].mean()
)

n_perms = 1000
perm_diffs = []
for _ in range(n_perms):
    shuffled = analysis["outcome_t1"].sample(frac=1).values
    diff = analysis["gpss_z"][shuffled].mean() - analysis["gpss_z"][~shuffled].mean()
    perm_diffs.append(diff)

perm_p = np.mean([d >= observed_diff for d in perm_diffs])
print(f"   Observed mean diff: {observed_diff:.4f}")
print(f"   Permutation p-value: {perm_p:.4f}")
print(f"   {'✅ Signal survives placebo' if perm_p < 0.05 else '❌ Signal does not survive placebo'}")

# ── Validation 2: Inter-rater Reliability (rescore 10% sample) ────────────────
print("\n2. INTER-RATER RELIABILITY")
print("   → To run: re-score 10% of transcripts and compute ICC.")
print("   → Included as a stub; requires a second LLM pass (set RESCORE=True to run).")

RESCORE = False  # flip to True to actually rescore
if RESCORE:
    transcripts = pd.read_parquet(PROCESSED_DIR / "transcripts.parquet")
    sample = transcripts.sample(frac=0.1, random_state=42)
    rescore_records = []
    for _, row in sample.iterrows():
        # Score with slightly different temperature to test stability
        result = score_transcript(row["symbol"], row["quarter"] + "_v2", row["full_text"])
        if result:
            rescore_records.append(result)
    if rescore_records:
        rescore_df = pd.DataFrame(rescore_records)
        # Compute correlation between original and rescore
        merged = scores.merge(rescore_df, on=["symbol"], suffixes=("_v1", "_v2"))
        corr = merged["gpss_composite_v1"].corr(merged["gpss_composite_v2"])
        print(f"   Pearson correlation (v1 vs v2): {corr:.3f}")

# ── Validation 3: Sentiment Baseline Comparison ───────────────────────────────
print("\n3. SENTIMENT BASELINE COMPARISON")
print("   → Alpha Vantage transcripts include per-speaker sentiment scores.")
print("   → Computing mean transcript sentiment as a naive baseline.")

transcripts = pd.read_parquet(PROCESSED_DIR / "transcripts.parquet")
# Re-extract sentiment from raw cached files
sentiment_records = []
for _, row in transcripts.iterrows():
    cache = RAW_DIR / "transcripts" / f"{row['symbol']}_{row['quarter']}.json"
    if cache.exists():
        with open(cache) as f:
            raw = json.load(f)
        speakers = raw.get("transcript", [])
        if isinstance(speakers, list):
            sents = [float(s.get("sentiment", 0)) for s in speakers if s.get("sentiment")]
            if sents:
                sentiment_records.append({
                    "symbol": row["symbol"],
                    "quarter": row["quarter"],
                    "mean_sentiment": np.mean(sents),
                })

if sentiment_records:
    sent_df = pd.DataFrame(sentiment_records)
    merged = analysis.merge(sent_df, on=["symbol", "quarter"], how="left")
    merged = merged.dropna(subset=["mean_sentiment"])

    sent_miss = merged[merged["outcome_t1"]==True]["mean_sentiment"]
    sent_nomiss = merged[merged["outcome_t1"]==False]["mean_sentiment"]
    t_s, p_s = stats.ttest_ind(sent_miss, sent_nomiss, equal_var=False)

    print(f"   Mean sentiment: pre-MISS={sent_miss.mean():.3f}, pre-NO_MISS={sent_nomiss.mean():.3f}")
    print(f"   Sentiment t-test p-value: {p_s:.4f}")
    print(f"   GPSS t-test p-value:      {t_pval:.4f}")
    if t_pval < p_s:
        print("   ✅ GPSS provides stronger signal than raw sentiment")
    else:
        print("   ⚠️  Raw sentiment is comparable or stronger than GPSS")

## 12. Summary & Export

Export final analysis dataset and print summary statistics for the report.

In [ ]:
analysis = pd.read_parquet(PROCESSED_DIR / "analysis_dataset.parquet")

print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"\nDataset:")
print(f"  Tickers: {analysis['symbol'].nunique()}")
print(f"  Quarters: {analysis['quarter'].nunique()}")
print(f"  Total observations: {len(analysis)}")
print(f"  Pre-MISS: {analysis['outcome_t1'].sum()} ({100*analysis['outcome_t1'].mean():.1f}%)")
print(f"  Pre-NO_MISS: {(~analysis['outcome_t1']).sum()} ({100*(~analysis['outcome_t1']).mean():.1f}%)")
print(f"\nGPSS Composite:")
print(f"  Overall mean: {analysis['gpss_composite'].mean():.2f}")
print(f"  Overall std:  {analysis['gpss_composite'].std():.2f}")
print(f"  Pre-MISS mean: {analysis[analysis['outcome_t1']==True]['gpss_composite'].mean():.2f}")
print(f"  Pre-NO_MISS mean: {analysis[analysis['outcome_t1']==False]['gpss_composite'].mean():.2f}")

# Export clean CSV for report tables
export_cols = ["symbol", "quarter", "D1", "D2", "D3", "D4", "D5", "D6",
               "gpss_composite", "gpss_z", "outcome_t1"]
analysis[export_cols].to_csv(PROCESSED_DIR / "analysis_export.csv", index=False)
print(f"\n✅ Exported to data/processed/analysis_export.csv")

# Data manifest
print(f"\nAll output files:")
for f in sorted(PROCESSED_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size:>8.1f} KB")

---

## Appendix A: Repo Structure

When deployed, the repo should look like:

```
gpss-pipeline/
├── gpss_pipeline.ipynb          # This notebook (main pipeline)
├── requirements.txt             # Python dependencies
├── .env.example                 # Template for API keys
├── .gitignore                   # Ignore data/, .env
├── README.md                    # Project overview
└── data/
    ├── raw/                     # Cached API responses (gitignored)
    │   ├── transcripts/
    │   ├── earnings/
    │   └── prices/
    ├── processed/               # Parquet files + analysis outputs
    │   ├── earnings.parquet
    │   ├── transcripts.parquet
    │   ├── prices.parquet
    │   ├── labels.parquet
    │   ├── gpss_scores.parquet
    │   ├── analysis_dataset.parquet
    │   ├── analysis_export.csv
    │   └── gpss_results.png
    └── gpss_scores/             # Cached LLM scoring results
```

## Appendix B: Claude Code Deployment Instructions

To deploy this pipeline from Claude Code:

```bash
# 1. Create repo
mkdir gpss-pipeline && cd gpss-pipeline
git init

# 2. Create supporting files
echo 'requests\npandas\nnumpy\nscipy\nstatsmodels\nmatplotlib\nseaborn\ntqdm\nanthropic' > requirements.txt

echo 'ALPHA_VANTAGE_API_KEY=your_key_here\nANTHROPIC_API_KEY=your_key_here' > .env.example

echo 'data/\n.env\n__pycache__/\n*.pyc\n.ipynb_checkpoints/' > .gitignore

# 3. Copy notebook into repo
cp /path/to/gpss_pipeline.ipynb .

# 4. Install dependencies
pip install -r requirements.txt

# 5. Set API keys
cp .env.example .env
# Edit .env with real keys, or export as environment variables:
export ALPHA_VANTAGE_API_KEY=your_key
export ANTHROPIC_API_KEY=your_key

# 6. Run the notebook
jupyter notebook gpss_pipeline.ipynb
# Or headless: jupyter nbconvert --execute gpss_pipeline.ipynb
```

## Appendix C: Rate Limit Strategy

| Tier | Limit | Time to ingest 12 tickers × 20 quarters |
|------|-------|------------------------------------------|
| Free | 25 req/day | ~10 days (transcripts are the bottleneck) |
| Paid ($50/mo) | 75 req/min | ~30 minutes |

The pipeline is designed for **incremental ingestion**: every API response is cached to disk. If you hit the daily limit, just re-run the cell the next day — it picks up where it left off.

## Appendix D: Extending the Pipeline

To add a 7th rubric dimension, only one change is needed:
1. Add the dimension description to `GPSS_SYSTEM_PROMPT`
2. Add the key to the validation and composite calculation

No retraining, no feature engineering — this is the core advantage of the LLM-as-Judge approach.